<a href="https://colab.research.google.com/github/ankorn/arcanumsearch/blob/main/arc_retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip -q install sentence-transformers[train] datasets faiss-gpu-cu12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 47.5 MB/s eta 0:00:00:00:0100:01


In [2]:
from datasets import load_dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.losses import GISTEmbedLoss, MultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sentence_transformers.util import mine_hard_negatives
import torch

/tmp/ipykernel_1655/1708220275.py:7: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers.losses import GISTEmbedLoss, MultipleNegativesRankingLoss
/tmp/ipykernel_1655/1708220275.py:8: DeprecationWarning: Importing from 'sentence_transformers.training_args' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.training_args' instead.
  from sentence_transformers.training_args import BatchSamplers
/tmp/ipykernel_1655/1708220275.py:9: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import InformationRetrievalEvaluator


In [3]:
dataset = load_dataset("pameydorke/arcanum-quests-queries-synthetic-v2")
train_dataset = dataset["train"]

train_dataset = train_dataset.rename_columns({
    "query": "anchor",
    "enhanced_document_text": "positive"
})

columns_to_keep = ["anchor", "positive"]
train_dataset = train_dataset.select_columns(columns_to_keep)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/632 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/224k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/158k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/912 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/228 [00:00<?, ? examples/s]

In [4]:
model = SentenceTransformer('BAAI/bge-m3')
guide_model = SentenceTransformer('BAAI/bge-m3')

loss = GISTEmbedLoss(
    model=model,
    guide=guide_model,
    temperature=0.01
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [5]:
test_dataset = dataset["test"]

queries = {}
corpus = {}
relevant_docs = {}

unique_docs = {}
for idx, example in enumerate(test_dataset):
    doc_text = example['enhanced_document_text']
    if doc_text not in unique_docs:
        doc_id = f"doc_{len(unique_docs)}"
        unique_docs[doc_text] = doc_id
        corpus[doc_id] = doc_text

for idx, example in enumerate(test_dataset):
    query = example['query']
    doc_text = example['enhanced_document_text']
    doc_id = unique_docs[doc_text]
    query_id = f"query_{idx}"
    queries[query_id] = query
    relevant_docs[query_id] = {doc_id}

evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name="arcanum-test",
    show_progress_bar=False,
)

In [6]:
training_args = SentenceTransformerTrainingArguments(
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    # gradient_accumulation_steps=2,
    warmup_ratio=0.1,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="steps",
    save_total_limit=2,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
)


In [7]:
trainer = SentenceTransformerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    loss=loss,
    evaluator=evaluator,
)
trainer.train()

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss,Validation Loss,Arcanum-test Cosine Accuracy@1,Arcanum-test Cosine Accuracy@3,Arcanum-test Cosine Accuracy@5,Arcanum-test Cosine Accuracy@10,Arcanum-test Cosine Precision@1,Arcanum-test Cosine Precision@3,Arcanum-test Cosine Precision@5,Arcanum-test Cosine Precision@10,Arcanum-test Cosine Recall@1,Arcanum-test Cosine Recall@3,Arcanum-test Cosine Recall@5,Arcanum-test Cosine Recall@10,Arcanum-test Cosine Ndcg@10,Arcanum-test Cosine Mrr@10,Arcanum-test Cosine Map@100
10,0.491211,No log,0.736842,0.921053,0.964912,0.982456,0.736842,0.307018,0.192982,0.098246,0.736842,0.921053,0.964912,0.982456,0.869797,0.832320,0.833642
20,0.210485,No log,0.732456,0.921053,0.973684,0.995614,0.732456,0.307018,0.194737,0.099561,0.732456,0.921053,0.973684,0.995614,0.871717,0.830780,0.831117
30,0.269731,No log,0.728070,0.903509,0.964912,0.995614,0.728070,0.301170,0.192982,0.099561,0.728070,0.903509,0.964912,0.995614,0.864899,0.822323,0.822492
40,0.394795,No log,0.714912,0.907895,0.964912,0.978070,0.714912,0.302632,0.192982,0.097807,0.714912,0.907895,0.964912,0.978070,0.853695,0.812622,0.814158
50,0.277195,No log,0.714912,0.894737,0.960526,0.986842,0.714912,0.298246,0.192105,0.098684,0.714912,0.894737,0.960526,0.986842,0.856503,0.813804,0.814480
60,0.391223,No log,0.732456,0.912281,0.960526,0.978070,0.732456,0.304094,0.192105,0.097807,0.732456,0.912281,0.960526,0.978070,0.864491,0.826803,0.828298
70,0.154496,No log,0.697368,0.903509,0.947368,0.982456,0.697368,0.301170,0.189474,0.098246,0.697368,0.903509,0.947368,0.982456,0.846149,0.801507,0.802449
80,0.224641,No log,0.714912,0.907895,0.960526,0.991228,0.714912,0.302632,0.192105,0.099123,0.714912,0.907895,0.960526,0.991228,0.860543,0.817426,0.817844
90,0.134883,No log,0.723684,0.921053,0.978070,0.991228,0.723684,0.307018,0.195614,0.099123,0.723684,0.921053,0.978070,0.991228,0.869258,0.828599,0.829006
100,0.092454,No log,0.728070,0.916667,0.964912,0.991228,0.728070,0.305556,0.192982,0.099123,0.728070,0.916667,0.964912,0.991228,0.867668,0.826960,0.827276


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=171, training_loss=0.18606704427065512, metrics={'train_runtime': 371.0107, 'train_samples_per_second': 7.374, 'train_steps_per_second': 0.461, 'total_flos': 0.0, 'train_loss': 0.18606704427065512, 'epoch': 3.0})

In [ ]:
import huggingface_hub

huggingface_hub.login()

In [10]:
model.push_to_hub("pameydorke/arcanum-quests-base-retriever-v3")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp7l_oza8b/tokenizer.json:  48%|####7     | 7.98MB / 16.8MB            

  ...l_oza8b/model.safetensors:   0%|          | 7.21MB / 2.27GB            

'https://huggingface.co/pameydorke/arcanum-quests-base-retriever-v3/commit/e98537515ccfa82ff11f7ccfa22302dd5d9ef6a0'

In [4]:
BASE_MODEL_NAME = 'pameydorke/arcanum-quests-base-retriever-v3'

In [5]:
model = SentenceTransformer(BASE_MODEL_NAME)

modules.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/283 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/241 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/380 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.8M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

In [13]:
loss = MultipleNegativesRankingLoss(model=model)

In [6]:
# train_dataset_hard = mine_hard_negatives(
#     dataset=train_dataset,
#     model=model,
#     range_min=10,
#     range_max=100,
#     max_score=0.9,
#     num_negatives=3,
#     batch_size=16,
#     use_faiss=True,
# )
train_dataset_hard_default = mine_hard_negatives(
    dataset=train_dataset,
    model=model,
    num_negatives=1,
    range_min=10,
    range_max=50,
    sampling_strategy="top",
    use_faiss=True,
)

Found 907 unique queries out of 912 total queries.
Found an average of 1.006 positives per query.


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/29 [00:00<?, ?it/s]

Querying FAISS index: 100%|██████████| 1/1 [00:00<00:00,  7.22it/s]


Negative candidates mined, preparing dataset...
Metric       Positive       Negative     Difference
Count             912            912               
Mean           0.6620         0.4447         0.2173
Median         0.6614         0.4434         0.2142
Std            0.0674         0.0322         0.0681
Min            0.4528         0.3441         0.0061
25%            0.6139         0.4227         0.1694
50%            0.6615         0.4435         0.2143
75%            0.7084         0.4643         0.2634
Max            0.8498         0.5498         0.4240


In [34]:
import torch.nn.functional as F

for i, row in train_dataset_hard_default.to_pandas().iterrows():
    if i < 3:
        continue
    
    if i > 5:
        break
    
    query_e = model.encode(row['anchor'], convert_to_tensor=True).unsqueeze(0)
    positive_e = model.encode(row['positive'], convert_to_tensor=True).unsqueeze(0)
    negative_e = model.encode(row['negative'], convert_to_tensor=True).unsqueeze(0)
    
    positive_score = F.cosine_similarity(query_e, positive_e).item()
    negative_score = F.cosine_similarity(query_e, negative_e).item()
    
    print(row['anchor'], '\n')
    print('P:', f"{positive_score:.2f}", ' | ', row['positive'], '\n')
    print('N:',f"{negative_score:.2f}", ' | ',  row['negative'], '\n')
    
    print('>' * 50, '\n', '\n', '\n')
    

strongbox steafast anvill 

P: 0.62  |  The Innkeeper's Strongbox. Drop Yer Anchor Daniel Hallaway can be found in Black Root at his Inn Talking with him reveals the high prices for staying at his Inn If Hallaway has a Reaction Modifier of at least 41 then The Player can ask for a cheaper alternative Hallaway then further reveals that he has a family Strongbox at The Steadfast Anvil, with difficulties getting it back Hallaway will offer the quest to get his Strongbox back In return the player can stay at his Inn for free forever The Steadfast Anvil Garret Almstead can be found in Black Root at his Smithy Talking with him reveals that Hallaway brought the Stronbox to him for repair It comes to light that there is a difference in expectations between the two men over what repairs were required This can be resolved by Convince him to return the box free (CH 15) Pay him for the box, prices vary based on (CH 7-14) Lock pick The Chest that has the Strongbox Pick pocket Garret and use the Key

In [20]:
training_args = SentenceTransformerTrainingArguments(
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    fp16=True,
    warmup_ratio=0.1,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="steps",
    save_total_limit=1,
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_hard,
    loss=loss,
    evaluator=evaluator,
)
trainer.train()

Step,Training Loss,Validation Loss,Arcanum-test Cosine Accuracy@1,Arcanum-test Cosine Accuracy@3,Arcanum-test Cosine Accuracy@5,Arcanum-test Cosine Accuracy@10,Arcanum-test Cosine Precision@1,Arcanum-test Cosine Precision@3,Arcanum-test Cosine Precision@5,Arcanum-test Cosine Precision@10,Arcanum-test Cosine Recall@1,Arcanum-test Cosine Recall@3,Arcanum-test Cosine Recall@5,Arcanum-test Cosine Recall@10,Arcanum-test Cosine Ndcg@10,Arcanum-test Cosine Mrr@10,Arcanum-test Cosine Map@100
10,0.373538,No log,0.728070,0.925439,0.960526,0.986842,0.728070,0.308480,0.192105,0.098684,0.728070,0.925439,0.960526,0.986842,0.870462,0.831687,0.832514
20,0.261381,No log,0.719298,0.907895,0.956140,0.986842,0.719298,0.302632,0.191228,0.098684,0.719298,0.907895,0.956140,0.986842,0.860728,0.819333,0.820191
30,0.247653,No log,0.697368,0.907895,0.973684,0.986842,0.697368,0.302632,0.194737,0.098684,0.697368,0.907895,0.973684,0.986842,0.857507,0.814303,0.815332
40,0.224824,No log,0.719298,0.934211,0.964912,0.991228,0.719298,0.311404,0.192982,0.099123,0.719298,0.934211,0.964912,0.991228,0.865748,0.824196,0.824854
50,0.232279,No log,0.719298,0.907895,0.969298,0.982456,0.719298,0.302632,0.193860,0.098246,0.719298,0.907895,0.969298,0.982456,0.862286,0.822198,0.823698
60,0.217463,No log,0.714912,0.916667,0.964912,0.986842,0.714912,0.305556,0.192982,0.098684,0.714912,0.916667,0.964912,0.986842,0.861873,0.820405,0.821372
70,0.221214,No log,0.692982,0.912281,0.956140,0.991228,0.692982,0.304094,0.191228,0.099123,0.692982,0.912281,0.956140,0.991228,0.853424,0.807937,0.808428
80,0.199311,No log,0.701754,0.916667,0.969298,0.986842,0.701754,0.305556,0.193860,0.098684,0.701754,0.916667,0.969298,0.986842,0.859473,0.816891,0.817465
90,0.162097,No log,0.710526,0.916667,0.969298,0.991228,0.710526,0.305556,0.193860,0.099123,0.710526,0.916667,0.969298,0.991228,0.862282,0.819596,0.819937
100,0.178335,No log,0.701754,0.894737,0.951754,0.991228,0.701754,0.298246,0.190351,0.099123,0.701754,0.894737,0.951754,0.991228,0.855452,0.810970,0.811461


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [61]:
model.push_to_hub("pameydorke/arcanum-quests-retriever-with-hard-negatives")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpljhriwyp/tokenizer.json:  48%|####7     | 7.98MB / 16.8MB            

  ...jhriwyp/model.safetensors:   1%|1         | 24.7MB / 2.27GB            

'https://huggingface.co/pameydorke/arcanum-quests-retriever-with-hard-negatives/commit/49c85d3e99b6db7801e27bae5b4acdb598a61367'

In [ ]:
from sentence_transformers import SentenceTransformer
import torch
import torch.nn.functional as F

documents = list(dataset['train']['document_text']) + list(dataset['test']['document_text'])
links = list(dataset['train']['document_link']) + list(dataset['test']['document_link'])

doc_embeddings = model.encode(
    documents,
    convert_to_tensor=True,
    show_progress_bar=True,
    batch_size=32
)

def retrieve(query, top_k=5):
    query_embedding = model.encode(query, convert_to_tensor=True)

    similarities = F.cosine_similarity(query_embedding, doc_embeddings)

    scores, indices = torch.topk(similarities, k=top_k)

    return [
        {"score": float(scores[i]), "link": links[indices[i]], "text": documents[indices[i]]}
        for i in range(top_k)
    ]

Batches:   0%|          | 0/22 [00:00<?, ?it/s]

In [89]:
for t in ["bow master", "master of bow", "bow mastery quest", "azram star quest", "Strongbox quest"]:
  results = retrieve(t)
  for r in results:
      print(f"{r['score']:.4f} | {r['link']} | {r['text'][:100]}...")

  print('\n')

0.3788 | https://arcanum.fandom.com/wiki/Masters_of_Magick | You can become the Master Of Earth by talking with Addo Terrin and taking his test. This involves ge...
0.3788 | https://arcanum.fandom.com/wiki/Masters_of_Magick | You can become the Master Of Earth by talking with Addo Terrin and taking his test. This involves ge...
0.3788 | https://arcanum.fandom.com/wiki/Masters_of_Magick | You can become the Master Of Earth by talking with Addo Terrin and taking his test. This involves ge...
0.3783 | https://arcanum.fandom.com/wiki/Masters_of_Magick | You can become the Master of Air by talking with Wek’ K’ene and taking his test. This involves getti...
0.3783 | https://arcanum.fandom.com/wiki/Masters_of_Magick | You can become the Master of Air by talking with Wek’ K’ene and taking his test. This involves getti...
0.3783 | https://arcanum.fandom.com/wiki/Masters_of_Magick | You can become the Master of Air by talking with Wek’ K’ene and taking his test. This involves getti...
0.3588 | h